# Encoding en JSON

Ya sabemos que un archivo de texto son bytes y que el encoding dice cómo traducirlos.
Hoy: qué pasa con las tildes y eñes cuando guardamos y leemos JSON.

> Usamos los mismos datos que la clase anterior: `datos_encoding.zip` descomprimido junto a este notebook.

## Repaso: de texto a bytes y de vuelta

In [28]:
texto = "Ñorquinco"

b = texto.encode("utf-8")
print(type(b),list(b))


<class 'bytes'> [195, 145, 111, 114, 113, 117, 105, 110, 99, 111]


In [19]:
print(b.decode("utf-8"))

Ñorquinco


In [21]:
print(b.decode("cp1252"))

Ã‘orquinco


In [29]:
bytes([200,32]).decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc8 in position 0: invalid continuation byte

In [31]:
bytes([195]).decode('cp1252'),bytes([195]).decode('latin-1')

('Ã', 'Ã')

In [27]:
bytes([200,97]).decode("cp1252")

'Èa'

## El formato Unicode

Cada carácter tiene un número (*code point*). Se suele escribir en hexadecimal con el formato `U+XXXX`:

In [32]:
import unicodedata

for c in "Aé°Ñ≈🌧":
    print(f"{c}  {ord(c):6}   {unicodedata.name(c)}")

A      65   LATIN CAPITAL LETTER A
é     233   LATIN SMALL LETTER E WITH ACUTE
°     176   DEGREE SIGN
Ñ     209   LATIN CAPITAL LETTER N WITH TILDE
≈    8776   ALMOST EQUAL TO
🌧  127783   CLOUD WITH RAIN


### Secuencias de escape

Dentro de un string, la barra invertida `\` es el carácter de escape: avisa que lo que sigue no se toma literal.
La barra junto con lo que le sigue forman una secuencia de escape, que vale por un solo carácter.



In [33]:
print("Estación:\tBariloche\nTemperatura:\t8.5")
print(len("\n"), len("\t"), len("\\"))
print("Dijo \"hola\"")

Estación:	Bariloche
Temperatura:	8.5
1 1 1
Dijo "hola"


Como la barra es especial, para ver una barra sola hay que escaparla: por eso `repr` la muestra doble.

In [34]:
barra = "\\"
print(barra)
print(repr(barra))
print(len(barra))

\
'\\'
1


**Trampa clásica con rutas de Windows:** el `\n` de `\nuevo` se convierte en salto de línea.

In [35]:
print("C:\nuevo\datos")
print(r"C:\nuevo\datos")   # raw string: las barras se toman literales

C:
uevo\datos
C:\nuevo\datos


<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/tmp/ipykernel_91556/4058746958.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  print("C:\nuevo\datos")


Además Python avisa con un `SyntaxWarning`: `\d` no es una secuencia de escape válida 

Otra razón para construir rutas con `Path("C:/nuevo/datos")` o `Path("C:") / "nuevo" / "datos"`.

## `json.dumps` y los caracteres no ASCII

`json.dumps` convierte un diccionario en un `str` con formato JSON. Miremos qué hace con las tildes:

In [36]:
import json

medicion = {"estacion": "Neuquén", "temperatura": 19.2, "unidad": "°C"}

print(json.dumps(medicion))

{"estacion": "Neuqu\u00e9n", "temperatura": 19.2, "unidad": "\u00b0C"}


Por defecto, `json` escapa todo lo que no es ASCII: la `é` aparece como `\u00e9`.

In [38]:
print(ord("é"))
print(hex(ord("é")))
print(f"{ord('é'):04x}")

233
0xe9
00e9


## argumento nombrado en los métodos de json: `ensure_ascii` 

In [37]:
print(json.dumps(medicion, ensure_ascii=False))

{"estacion": "Neuquén", "temperatura": 19.2, "unidad": "°C"}


In [39]:
print(json.dumps(medicion, ensure_ascii=False, indent=2))

{
  "estacion": "Neuquén",
  "temperatura": 19.2,
  "unidad": "°C"
}


Los dos son JSON válidos y representan **exactamente los mismos datos**:

## Guardar JSON en un archivo

El estándar de JSON (RFC 8259) pide **UTF-8**. Entonces:
- `ensure_ascii=False` para que se lea bien,
- `encoding="utf-8"` en `open()` para que los bytes sean UTF-8.

In [44]:
from pathlib import Path

salida = Path("salida")
salida.mkdir(parents=True, exist_ok=True)

ruta = salida / "medicion.json"

with open(ruta, "w", encoding="utf-8") as archivo:
    json.dump(medicion, archivo, ensure_ascii=False, indent=2)

with open(ruta, "r", encoding="utf-8") as archivo:
    print(archivo.read())

{
  "estacion": "Neuquén",
  "temperatura": 19.2,
  "unidad": "°C"
}


Y los bytes que quedaron en el disco:

In [45]:
with open(ruta, "rb") as archivo:
    print(archivo.read())

b'{\n  "estacion": "Neuqu\xc3\xa9n",\n  "temperatura": 19.2,\n  "unidad": "\xc2\xb0C"\n}'


## ¿Qué pasa si no indicamos el encoding?

`open()` usa el del sistema: en Windows, `cp1252`. Simulémoslo poniéndolo explícito:

In [46]:
ruta_win = salida / "medicion_windows.json"

with open(ruta_win, "w", encoding="cp1252") as archivo:
    json.dump(medicion, archivo, ensure_ascii=False, indent=2)

with open(ruta_win, "rb") as archivo:
    print(archivo.read())

b'{\n  "estacion": "Neuqu\xe9n",\n  "temperatura": 19.2,\n  "unidad": "\xb0C"\n}'


La `é` quedó como `\xe9` (un byte, cp1252). Ese archivo **no es un JSON estándar**.
Si otra persona (o un programa en Linux, o una página web) lo lee como UTF-8:

In [ ]:
with open(ruta_win, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

Y si los datos tienen algo que no existe en cp1252, ni siquiera se puede guardar:

In [ ]:
nota = {"observacion": "CO₂ ≈ 420 ppm"}

with open(salida / "nota.json", "w", encoding="cp1252") as archivo:
    json.dump(nota, archivo, ensure_ascii=False)

### ¿Y con los escapes?

Con `ensure_ascii=True` (el default), el texto es ASCII puro: **cualquier encoding** lo guarda igual.

In [ ]:
for enc in ["utf-8", "cp1252"]:
    print(enc, json.dumps(nota).encode(enc))

## Leer JSON

Al leer, siempre `encoding="utf-8"`. Los escapes `\uXXXX` se convierten solos:

In [ ]:
with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

print(datos)
print(datos["estacion"])

In [ ]:
json.loads('{"estacion": "Neuqu\\u00e9n"}')

---
# Ejercicio

1. Recorrer todos los `.txt` de `datos/smn/`.
2. Leer cada uno detectando si está en `utf-8` o `cp1252`.
3. Guardar un único `salida/observaciones.json` con todas las observaciones, legible (`ensure_ascii=False`, `indent=2`) y en UTF-8.
4. Volver a abrir el JSON y mostrar las estaciones sin repetir.

**Verificación:** `Ñorquinco`, `Neuquén` y `Río Gallegos` tienen que aparecer bien escritos.

In [ ]:
# Tu solución acá